# Langfuse Demo — Chapter 5: OpenTelemetry Auto-Instrumentation

---

## The problem with manual instrumentation

In the previous notebooks, every LLM call required a wrapper:

```python
# Manual approach (Chapter 1–4 style)
with langfuse.start_as_current_observation(as_type="generation", name="llm-answer", input=messages) as gen:
    answer, usage = call_llm(messages)
    gen.update(
        model=settings.groq_model,
        output=answer,
        usage_details={"input": usage["input"], "output": usage["output"]},
        cost_details={"input": usage["input_cost"], "output": usage["output_cost"]},
    )
```

In a large codebase with dozens of LLM calls, this boilerplate accumulates fast.

## The solution: OpenTelemetry + OpenInference

`GroqInstrumentor` patches the Groq SDK at the OpenTelemetry level — **one line of setup,
then every Groq API call is automatically traced** as a generation:

```python
# OTel approach — setup once
from openinference.instrumentation.groq import GroqInstrumentor
GroqInstrumentor().instrument()

# Then just call Groq — tokens, model, latency, input/output captured automatically
response = groq_client.chat.completions.create(model=..., messages=...)
```

## What gets auto-captured vs what still needs manual code

| Field | OTel Auto | Still Manual |
|-------|:---------:|:------------:|
| Model name | ✅ | |
| Input messages | ✅ | |
| Output text | ✅ | |
| Input / output tokens | ✅ | |
| LLM call latency | ✅ | |
| Cost (USD) | | ✅ OTel doesn't know Groq pricing |
| `user_id` / `session_id` | | ✅ `propagate_attributes` |
| Retrieval / reranking spans | | ✅ `start_as_current_observation` |
| Span metadata | | ✅ `span.update(metadata=...)` |
| Scores | | ✅ `langfuse.create_score(...)` |

In [8]:
from dotenv import load_dotenv
from groq import Groq
from langfuse import propagate_attributes
from openinference.instrumentation.groq import GroqInstrumentor

from langfuse_demo.client import get_client
from langfuse_demo.config import settings

_ = load_dotenv()

langfuse = get_client()
assert langfuse.auth_check()

# One-time setup: patches the Groq SDK globally via OpenTelemetry
# After this, every groq_client.chat.completions.create() call is auto-traced
GroqInstrumentor().instrument()

# Use Groq directly (not the call_llm wrapper) so auto-instrumentation is visible
groq_client = Groq(api_key=settings.groq_api_key)

print(f"Langfuse connected: {settings.langfuse_host}")
print("GroqInstrumentor active — all Groq calls now auto-traced")

Attempting to instrument while already instrumented


Langfuse connected: http://localhost:3000
GroqInstrumentor active — all Groq calls now auto-traced


---

## Part 1: Minimal Example

The simplest possible auto-traced call. No wrappers. No `gen.update()`. No `usage_details`.

Compare this with `nb01-manual-trace` to see what disappears.

In [7]:
from langfuse_demo.llm import GROQ_PRICE_INPUT, GROQ_PRICE_OUTPUT

# No start_as_current_observation. No gen.update(). No usage_details.
# GroqInstrumentor handles model, tokens, latency, input/output automatically.
response = groq_client.chat.completions.create(
    model=settings.groq_model,
    messages=[
        {"role": "system", "content": "You are a concise ML expert."},
        {"role": "user", "content": "What is gradient descent? Answer in 2 sentences."},
    ],
    max_tokens=128,
)
answer = response.choices[0].message.content

# Cost is NOT auto-calculated by OTel (it doesn't know Groq pricing).
# Compute manually from the response object if you need it.
input_cost = round(response.usage.prompt_tokens * GROQ_PRICE_INPUT, 8)
output_cost = round(response.usage.completion_tokens * GROQ_PRICE_OUTPUT, 8)

langfuse.flush()

print(f"Answer: {answer}")
print(f"\nTokens: input={response.usage.prompt_tokens} | output={response.usage.completion_tokens}")
print(f"Cost: input=${input_cost:.6f} | output=${output_cost:.6f}  ← compute manually, OTel won't auto-fill this")
print(f"\nView traces: {settings.langfuse_host}/traces")
print("\nIn the UI: generation shows model, tokens, latency — zero extra code. Cost column empty (Groq not in built-in models).")

Answer: Gradient descent is an optimization algorithm used in machine learning to minimize the loss function of a model by iteratively adjusting its parameters in the direction of the negative gradient of the loss. This process reduces the loss and improves the model's performance by converging to a minimum point, allowing the model to make more accurate predictions.

Tokens : input=53 | output=64
Cost   : input=$0.000031 | output=$0.000051  ← compute manually, OTel won't auto-fill this

View traces: http://localhost:3000/traces

In the UI: generation shows model, tokens, latency — zero extra code. Cost column empty (Groq not in built-in models).


---

## Part 2: RAG Pipeline with Mixed Instrumentation

Real pipelines have both LLM calls and non-LLM steps (retrieval, reranking).
The pattern:

- **LLM calls** → auto-traced by `GroqInstrumentor` (no wrappers)
- **Non-LLM steps** → still need `start_as_current_observation` spans
- **User context** → still needs `propagate_attributes`

```
Trace
└── Span: rag-pipeline          ← manual (root)
      ├── Span: retrieval        ← manual (non-LLM)
      ├── Span: reranking        ← manual (non-LLM)
      └── Generation: groq/...  ← AUTO (GroqInstrumentor)
```

Notice: the generation name is now assigned by OTel (usually the model name or span name from the instrumentor),
not a custom string we define.

In [4]:
from langfuse_demo.rag import retrieve, rerank, format_context


def rag_pipeline_otel(
    question: str,
    user_id: str = "anonymous",
    session_id: str | None = None,
    top_k: int = 3,
) -> dict:
    """RAG pipeline using OTel auto-instrumentation for LLM calls."""

    with propagate_attributes(
        user_id=user_id,
        session_id=session_id,
        tags=["rag", "otel", "demo-ch05"],
    ):
        with langfuse.start_as_current_observation(
            as_type="span",
            name="rag-pipeline",
            input={"question": question},
        ) as pipeline:
            trace_id = langfuse.get_current_trace_id()

            # ── Step 1: Retrieval — still manual (not a Groq call) ────
            with langfuse.start_as_current_observation(
                as_type="span",
                name="retrieval",
                input={"query": question, "top_k": top_k},
            ) as retrieval_span:
                retrieved = retrieve(question, top_k=top_k)
                retrieval_span.update(
                    output={"docs": [d["id"] for d in retrieved]},
                    metadata={"retriever": "bm25-inMemory"},
                )

            # ── Step 2: Reranking — still manual (not a Groq call) ───
            with langfuse.start_as_current_observation(
                as_type="span",
                name="reranking",
                input={"n_docs": len(retrieved)},
            ) as rerank_span:
                reranked = rerank(question, retrieved)
                rerank_span.update(
                    output={"top_doc": reranked[0]["title"]},
                    metadata={"reranker": "title-overlap-boost"},
                )

            # ── Step 3: LLM — AUTO-TRACED by GroqInstrumentor ────────
            # No gen wrapper. No gen.update(). No usage_details.
            context = format_context(reranked)
            messages = [
                {
                    "role": "system",
                    "content": (
                        "You are a technical assistant for the Data Science team. "
                        "Answer using ONLY the provided context. "
                        "If the answer is not in the context, say 'I could not find that information in the documents.'"
                    ),
                },
                {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
            ]
            response = groq_client.chat.completions.create(
                model=settings.groq_model,
                messages=messages,
                max_tokens=512,
            )
            answer = response.choices[0].message.content

            pipeline.update(output={"answer": answer})

    return {
        "trace_id": trace_id,
        "answer": answer,
        "docs_used": [d["id"] for d in reranked],
        "tokens": {
            "input": response.usage.prompt_tokens,
            "output": response.usage.completion_tokens,
        },
    }

### Run questions and observe in the UI

In [5]:
questions = [
    ("What are the steps to set up the development environment?", "ana.silva", "otel-demo-session"),
    ("Which databases does the system use?", "joao.costa", "otel-demo-session"),
    ("What is a Trace in Langfuse?", "carla.mendes", "otel-demo-session"),
]

for question, user_id, session_id in questions:
    result = rag_pipeline_otel(question, user_id=user_id, session_id=session_id)
    print(f"Q: {question}")
    print(f"A: {result['answer'][:200]}..." if len(result["answer"]) > 200 else f"A: {result['answer']}")
    print(f"Tokens: input={result['tokens']['input']} | output={result['tokens']['output']}")
    print(f"{settings.langfuse_host}/trace/{result['trace_id']}")
    print()

langfuse.flush()
print(f"All traces flushed. View at: {settings.langfuse_host}/traces")

Q: What are the steps to set up the development environment?
A: The steps to set up the local development environment are as follows:

1. Prerequisites: Ensure Python 3.12+, uv, and Docker Desktop are installed.
2. Clone the repository: Run `git clone https://gith...
   Tokens: input=408 | output=116
   🔗 http://localhost:3000/trace/59b8930e6a6f2d58b72e8cf050f29886

Q: Which databases does the system use?
A: The system uses the following databases:

1. PostgreSQL 16 (for transactional and relational data)
2. ClickHouse (for analytics and high-volume logs)
3. Redis (for L2 cache and background job queues)
   Tokens: input=401 | output=46
   🔗 http://localhost:3000/trace/4001f9f6c3190b3a79bff103bbf11a98

Q: What is a Trace in Langfuse?
A: A Trace in Langfuse represents a complete interaction, such as a user question.
   Tokens: input=405 | output=17
   🔗 http://localhost:3000/trace/3381ae792f416b87e17a104e269d9a5c

✅ All traces flushed. View at: http://localhost:3000/traces


---

## Chapter 5 Summary

### Code reduction: before vs after

| | Manual SDK (Ch 1–4) | OTel (Ch 5) |
|-|---------------------|-------------|
| LLM call instrumentation | `start_as_current_observation` + `gen.update(usage_details=..., cost_details=...)` | Nothing — auto |
| Lines of code per LLM call | ~6–8 lines | 0 extra lines |
| Setup cost | 0 | `GroqInstrumentor().instrument()` once |
| Cost (USD) in Langfuse | ✅ via `cost_details=` | ❌ OTel doesn't know Groq pricing |
| Non-LLM spans | `start_as_current_observation` | Same |
| User context | `propagate_attributes` | Same |
| Scores | `langfuse.create_score` | Same |

### Cost in OTel pipelines

OTel captures token counts automatically. To get USD cost, compute it yourself from `response.usage` and either:
- Register the custom Groq model via `langfuse.api.models.create()` — Langfuse then infers cost from tokens (see Demo 05)
- Or track cost as custom metadata on a parent span

### When to use each approach

| Scenario | Recommended |
|----------|-------------|
| New project, clean slate | **OTel** — zero boilerplate for LLM calls |
| Existing codebase with many LLM calls | **OTel** — one-line migration, drop gen wrappers |
| Need USD cost in generation | **Manual SDK** — pass `cost_details=` explicitly |
| Need custom generation name / metadata on LLM span | **Manual SDK** — OTel span name comes from instrumentor |
| Non-LLM steps (retrieval, DB, custom logic) | **Manual SDK** — OTel only patches Groq |
| Both (hybrid) | Mix freely — OTel for LLM, manual for the rest |

### Key API

```python
from openinference.instrumentation.groq import GroqInstrumentor

GroqInstrumentor().instrument()   # enable auto-tracing
GroqInstrumentor().uninstrument() # disable (e.g. in tests)
```

**Package**: `openinference-instrumentation-groq`